← [01 · Finding and downloading data](01_finding_and_downloading_data.ipynb) · [Index](README.md) · [03 · Detrending and finding periods](03_detrending_and_finding_periods.ipynb) →
<!--nav-->

# 02 · From pixels to a light curve

So far a light curve has been handed to us as one flux number per timestamp. But a telescope is a **camera** — at each moment it records a little **image**. That single flux number is something *someone computed* from those images. Understanding how demystifies the data and shows you where errors can sneak in.

**You'll learn:** what a **Target Pixel File** is · how **aperture photometry** turns pixels into one number · why the choice of **aperture** changes your result.

## The Target Pixel File (TPF)

Downlinking a full image every 30 minutes for 150,000 stars is impossible, so Kepler saved only a small **postage stamp** of pixels around each target. That stack of little images over time is a **Target Pixel File (TPF)**.

Let's download one quarter of pixels for Kepler-8 and look at a single frame:

In [ ]:
import matplotlib.pyplot as plt

from skyplay import data, plotting

plotting.use_style()

# Wraps lk.search_targetpixelfile(...).download(). Not cached: lightkurve already
# caches the raw FITS, and a pixel cube is not a table.
tpf = data.load_tpf('kepler-8', quarter=4)

print('TPF shape (time, rows, cols):', tpf.shape)
tpf.plot(frame=100)
plt.show();

Each pixel has a brightness; the star is the bright blob near the center. Over the ~4000 frames in this file, the star's light spreads across several pixels (telescopes aren't perfectly sharp).

## Aperture photometry

To get *one* brightness number per frame, we pick a set of pixels that contains the star — the **aperture** — and **sum** them. Do that for every frame and you have a light curve. That's it. That's the whole idea.

The mission already chose a sensible aperture for each star, the **pipeline mask**. Let's see which pixels it selected:

In [ ]:
tpf.plot(aperture_mask=tpf.pipeline_mask)
plt.show();

The shaded pixels are the ones summed. Now convert to a light curve two different ways to see that **the aperture choice matters**:

- `'all'` — sum *every* pixel in the stamp (includes lots of empty sky → more background noise)
- `pipeline_mask` — sum only the star's pixels (cleaner)

In [ ]:
lc_all = tpf.to_lightcurve(aperture_mask='all').normalize()
lc_pipeline = tpf.to_lightcurve(aperture_mask=tpf.pipeline_mask).normalize()

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
lc_all.scatter(ax=ax[0], s=1, label='aperture = all pixels', c=plotting.SERIES[0])
lc_pipeline.scatter(ax=ax[1], s=1, label='aperture = pipeline mask', c=plotting.SERIES[1])
ax[0].set_title('Same star, two apertures -- note how much noisier "all pixels" is')
# Shared y-range, or the noisier panel quietly rescales and looks comparable.
ax[1].set_ylim(ax[0].get_ylim())
plt.show()

print('scatter (all)      :', round(lc_all.estimate_cdpp().value, 1), 'ppm-ish')
print('scatter (pipeline) :', round(lc_pipeline.estimate_cdpp().value, 1), 'ppm-ish')

The pipeline aperture is visibly cleaner: including empty sky pixels just adds background noise without adding star signal. Aperture choice also controls **contamination** — if a neighboring star falls inside your aperture, its light (and any variability) leaks into your measurement. This is a common source of false transit signals.

## Recap
- A **TPF** is a stack of small images over time.
- **Aperture photometry** = pick the star's pixels, sum them, repeat per frame → a light curve.
- The **aperture matters**: too big adds background noise; a badly placed one adds a neighbor's light. Real "discoveries" are often just contamination — always ask *which pixels made this number?*

## Learning resources
- 📗 [Lightkurve: what are TPFs?](https://docs.lightkurve.org/tutorials/1-getting-started/what-are-tpfs.html)
- 📗 [Lightkurve: aperture photometry](https://docs.lightkurve.org/tutorials/2-creating-light-curves/2-1-cutting-out-tpfs.html)
- 🌍 [Aperture photometry / photometry (astronomy)](https://en.wikipedia.org/wiki/Photometry_(astronomy))

**Next:** [`03_detrending_and_finding_periods.ipynb`](03_detrending_and_finding_periods.ipynb) — clean the curve and find the period automatically.